In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from src.model_evaluation import evaluate_model
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

In [11]:
dff = pd.read_pickle("../data/processed/model_df.pkl")
model_df=dff.copy()

In [100]:
model_df["is_high_risk"].value_counts(normalize=True)

is_high_risk
0    0.61892
1    0.38108
Name: proportion, dtype: float64

In [ ]:
model_df

,CustomerId,TotalTransactionValue,AverageTransactionValue,TransactionCount,StdTransactionValue,MaxTransactionValue,MinTransactionValue,is_high_risk
0,CustomerId_1,10000,10000.000000,1,0.000000,10000,10000,1
1,CustomerId_10,10000,10000.000000,1,0.000000,10000,10000,1
2,CustomerId_1001,30400,6080.000000,5,4100.243895,10000,200,1
3,CustomerId_1002,4775,434.090909,11,518.805446,1500,25,0
4,CustomerId_1003,32000,5333.333333,6,3945.461528,10000,1000,0
...,...,...,...,...,...,...,...,...
3737,CustomerId_992,32000,5333.333333,6,4033.195590,10000,1000,0
3738,CustomerId_993,32000,6400.000000,5,3781.534080,10000,1000,0
3739,CustomerId_994,614077,6079.970297,101,14537.733039,90000,10,0
3740,CustomerId_996,151000,8882.352941,17,2619.216317,10000,1000,1


## Separate Features and Target

In [12]:
X = model_df.drop(
    columns=[
        "CustomerId",
        "is_high_risk"
    ]
)

y = model_df["is_high_risk"]

## Train-Test Split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [113]:
training_pipeline = Pipeline([
    (
        "scaling",
        StandardScaler()
    )
])

In [16]:
X_train_processed = training_pipeline.fit_transform(
    X_train
)

X_test_processed = training_pipeline.transform(
    X_test
)

In [17]:
print("Training set:", X_train_processed.shape)
print("Testing set:", X_test_processed.shape)

print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

Training set: (2993, 6)
Testing set: (749, 6)
Training labels: (2993,)
Testing labels: (749,)


# Training a model
## Logistic Regression

In [ ]:
log_reg = LogisticRegression(
    random_state=42,
    max_iter=1000
)

log_reg.fit(
    X_train_processed,
    y_train
)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

## Random Forest

In [19]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(
    X_train_processed,
    y_train
)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

## Predictions

In [20]:
log_reg_pred = log_reg.predict(
    X_test_processed
)

rf_pred = rf_model.predict(
    X_test_processed
)

## Model Evaluation

In [ ]:
evaluate_model(
    y_test,
    log_reg_pred,
    "Logistic Regression"
)

evaluate_model(
    y_test,
    rf_pred,
    "Random Forest"
)


Logistic Regression
Accuracy: 0.6915887850467289
Precision: 0.6436170212765957
Recall: 0.4245614035087719
F1 Score: 0.5116279069767442

Random Forest
Accuracy: 0.7436582109479306
Precision: 0.6466876971608833
Recall: 0.7192982456140351
F1 Score: 0.6810631229235881


In [ ]:
log_auc = roc_auc_score(
    y_test,
    log_reg.predict_proba(
        X_test_processed
    )[:, 1]
)

rf_auc = roc_auc_score(
    y_test,
    rf_model.predict_proba(
        X_test_processed
    )[:, 1]
)

print("Logistic Regression AUC:", log_auc)
print("Random Forest AUC:", rf_auc)

Logistic Regression AUC: 0.3147005444646098
Random Forest AUC: 0.5956064730792499


c:\Users\bemnet\Desktop\credit-risk-model\credit-risk-model\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\Users\bemnet\Desktop\credit-risk-model\credit-risk-model\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


## Hyperparameter Tuning
### Grid Search

In [23]:
rf = RandomForestClassifier(
    random_state=42
)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, 15, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_search.fit(
    X_train_processed,
    y_train
)

print("Best Parameters:")
print(grid_search.best_params_)

print("Best ROC-AUC:")
print(grid_search.best_score_)

KeyboardInterrupt: 

### Random Search

In [24]:
rf = RandomForestClassifier(
    random_state=42
)

param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [5, 10, 15, 20, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2"]
}

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1
)

random_search.fit(
    X_train_processed,
    y_train
)

print("Best Parameters:")
print(random_search.best_params_)

print("Best ROC-AUC:")
print(random_search.best_score_)

Best Parameters:
{'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 15}
Best ROC-AUC:
0.798297402689388


In [25]:
best_rf = random_search.best_estimator_

In [26]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

rf_pred = best_rf.predict(
    X_test_processed
)

rf_prob = best_rf.predict_proba(
    X_test_processed
)[:, 1]

print("Accuracy:",
      accuracy_score(y_test, rf_pred))

print("Precision:",
      precision_score(y_test, rf_pred))

print("Recall:",
      recall_score(y_test, rf_pred))

print("F1 Score:",
      f1_score(y_test, rf_pred))

print("ROC-AUC:",
      roc_auc_score(y_test, rf_prob))

Accuracy: 0.7436582109479306
Precision: 0.6466876971608833
Recall: 0.7192982456140351
F1 Score: 0.6810631229235881
ROC-AUC: 0.806125226860254


# mlflow

In [49]:
import sys
print(sys.executable)

c:\Users\bemnet\Desktop\credit-risk-model\credit-risk-model\venv\Scripts\python.exe


In [88]:
import mlflow

mlflow.set_experiment(
    "Credit Risk Modeling"
)
print(
    mlflow.get_experiment_by_name(
        "Credit Risk Modeling"
    )
)

2026/06/04 12:33:11 INFO mlflow.tracking.fluent: Experiment with name 'Credit Risk Modeling' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/bemnet/Desktop/credit-risk-model/credit-risk-model/notebooks/mlruns/1', creation_time=1780565591724, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1780565591724, lifecycle_stage='active', name='Credit Risk Modeling', tags={}, trace_location=None, workspace='default'>


In [89]:
mlflow.set_tracking_uri(
    "sqlite:///notebooks/mlflow.db"
)

## Track Logistic Regression

In [90]:
import mlflow
import mlflow.sklearn

with mlflow.start_run(
    run_name="Logistic Regression"
):

    log_reg.fit(
        X_train_processed,
        y_train
    )

    predictions = log_reg.predict(
        X_test_processed
    )

    probabilities = (
        log_reg.predict_proba(
            X_test_processed
        )[:, 1]
    )

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions
    )

    recall = recall_score(
        y_test,
        predictions
    )

    f1 = f1_score(
        y_test,
        predictions
    )

    auc = roc_auc_score(
        y_test,
        probabilities
    )

    mlflow.log_param(
        "model",
        "LogisticRegression"
    )

    mlflow.log_metric(
        "accuracy",
        accuracy
    )

    mlflow.log_metric(
        "precision",
        precision
    )

    mlflow.log_metric(
        "recall",
        recall
    )

    mlflow.log_metric(
        "f1_score",
        f1
    )

    mlflow.log_metric(
        "roc_auc",
        auc
    )

    mlflow.sklearn.log_model(
        log_reg,
        "model"
    )

2026/06/04 12:33:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/04 12:33:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


## Track Baseline Random Forest

In [91]:
with mlflow.start_run(
    run_name="Random Forest Baseline"
):

    rf_model.fit(
        X_train_processed,
        y_train
    )

    predictions = rf_model.predict(
        X_test_processed
    )

    probabilities = (
        rf_model.predict_proba(
            X_test_processed
        )[:, 1]
    )

    mlflow.log_param(
        "model",
        "RandomForest"
    )

    mlflow.log_param(
        "n_estimators",
        100
    )

    mlflow.log_metric(
        "accuracy",
        accuracy_score(
            y_test,
            predictions
        )
    )

    mlflow.log_metric(
        "precision",
        precision_score(
            y_test,
            predictions
        )
    )

    mlflow.log_metric(
        "recall",
        recall_score(
            y_test,
            predictions
        )
    )

    mlflow.log_metric(
        "f1_score",
        f1_score(
            y_test,
            predictions
        )
    )

    mlflow.log_metric(
        "roc_auc",
        roc_auc_score(
            y_test,
            probabilities
        )
    )

    mlflow.sklearn.log_model(
        rf_model,
        "model"
    )

2026/06/04 12:33:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/04 12:33:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


## Track Tuned Random Forest

In [92]:
with mlflow.start_run(
    run_name="Random Forest Tuned"
):

    best_rf.fit(
        X_train_processed,
        y_train
    )

    predictions = best_rf.predict(
        X_test_processed
    )

    probabilities = (
        best_rf.predict_proba(
            X_test_processed
        )[:, 1]
    )

    mlflow.log_param(
        "model",
        "RandomForest"
    )

    mlflow.log_params(
        best_rf.get_params()
    )

    mlflow.log_metric(
        "accuracy",
        accuracy_score(
            y_test,
            predictions
        )
    )

    mlflow.log_metric(
        "precision",
        precision_score(
            y_test,
            predictions
        )
    )

    mlflow.log_metric(
        "recall",
        recall_score(
            y_test,
            predictions
        )
    )

    mlflow.log_metric(
        "f1_score",
        f1_score(
            y_test,
            predictions
        )
    )

    mlflow.log_metric(
        "roc_auc",
        roc_auc_score(
            y_test,
            probabilities
        )
    )

    mlflow.sklearn.log_model(
        best_rf,
        "model"
    )

2026/06/04 12:34:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/04 12:34:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [93]:
print(
    mlflow.get_tracking_uri()
)

sqlite:///notebooks/mlflow.db


In [94]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

for exp in client.search_experiments():
    print(
        f"ID={exp.experiment_id}, Name={exp.name}"
    )

ID=1, Name=Credit Risk Modeling
ID=0, Name=Default


## Model Selection Using MLflow

MLflow was used to track, compare, and manage all model training experiments. Each model run recorded the model type, hyperparameters, evaluation metrics, and model artifacts, enabling a structured and reproducible model development process.

Three model experiments were tracked:

1. Logistic Regression
2. Random Forest
3. Tuned Random Forest

### Model Performance Comparison

| Metric | Logistic Regression | Random Forest | Random Forest Tuned |
|----------|----------|----------|----------|
| Accuracy | 0.692 | **0.749** | 0.744 |
| Precision | 0.644 | **0.655** | 0.647 |
| Recall | 0.425 | **0.719** | **0.719** |
| F1-Score | 0.512 | **0.686** | 0.681 |
| ROC-AUC | 0.751 | 0.800 | **0.806** |

### Model Selection Criteria

The models were compared using multiple evaluation metrics, with particular emphasis on **ROC-AUC** and **Recall** because the objective of the project is to accurately identify high-risk customers.

Key findings:

- Logistic Regression provided an interpretable baseline but showed lower predictive performance.
- Random Forest significantly improved the identification of high-risk customers and achieved higher overall performance.
- The Tuned Random Forest achieved the highest ROC-AUC score (0.806) while maintaining strong recall performance (0.719), demonstrating the best ability to distinguish between high-risk and low-risk customers.

### Selected Model

Based on the MLflow experiment comparison, the **Tuned Random Forest** model was selected as the final model for credit risk prediction.

The selected model was registered in the **MLflow Model Registry** under the name **CreditRiskModel**, enabling model versioning, experiment reproducibility, and future deployment.

## Register the Model

In [86]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

for exp in client.search_experiments():
    print(
        f"ID={exp.experiment_id}, Name={exp.name}"
    )

ID=0, Name=Default


In [95]:
with mlflow.start_run(
    run_name="Final Random Forest Model"
):

    mlflow.sklearn.log_model(
        sk_model=best_rf,
        artifact_path="model",
        registered_model_name="CreditRiskModel"
    )

2026/06/04 12:34:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/06/04 12:34:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'CreditRiskModel'.
Created version '1' of model 'CreditRiskModel'.


In [96]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

for model in client.search_registered_models():
    print(model.name)

CreditRiskModel


In [97]:
MODEL_NAME = "CreditRiskModel"
model = mlflow.sklearn.load_model(
    f"models:/{MODEL_NAME}/latest"
)

In [99]:
print(mlflow.get_tracking_uri())

sqlite:///notebooks/mlflow.db


In [1]:
import mlflow

mlflow.set_tracking_uri(
    "sqlite:///notebooks/mlflow.db"
)

model = mlflow.sklearn.load_model(
    "models:/CreditRiskModel/latest"
)

print(type(model))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [2]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

for model in client.search_registered_models():
    print(model.name)

CreditRiskModel
